In [ ]:
##################################################################################################
# ATM 419/563
# mod_met_em_files.ipynb
# Megan Schiede, Fall 2026
#
# This script shows an example of how to modify met_em* files to modify initial meteorological 
# conditions after running submit_metgrid
#
#
# Will create a backup of original met_em* files for comparison
##################################################################################################

# Imports
import xarray as xr
import os

import matplotlib.pyplot as plt
from matplotlib import colors

import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
#path to your folder
path='./'

In [ ]:
# Flag to create backup met_em files
# Can be set to false after backup is created
write_backup = True

mod_temp = False
mod_rh = False

In [ ]:
## south_north grid point for index
sn_num=24
## west_east grid point for index
we_num=24
## num_metgrid_level point for index
mgl_num = 10

## color maps
temp_cmap = 'bwr'
rh_cmap = 'BrBG'

## Range for CB
# Temp.
temp_norm = colors.TwoSlopeNorm(vmin=-60, vcenter=0, vmax=10)
temp_norm_map = colors.TwoSlopeNorm(vmin=-60, vcenter=0, vmax=10) 
# RH
rh_norm = colors.TwoSlopeNorm(vmin=0, vcenter=50, vmax=100)
rh_norm_map = colors.Normalize(vmin=0, vmax=100)

## Map variables
proj = ccrs.PlateCarree()
lon_min = -83
lon_max = -65
lat_min = 36
lat_max = 49

In [ ]:
# Create backup met_em files
# Re-running submit_metgrid will return to original met_em files
if write_backup:
    for file in os.listdir(path):
        #Cycle through met_em files
        if file.startswith("met_em.d01."):
            file_path=os.path.join(path, file)

            #Import metgrid file as xarray dataset
            ds=xr.open_dataset(file_path)

            #save backup files as .nc files
            ds.to_netcdf(path+'backup_met_em.d01.'+str(ds.attrs['SIMULATION_START_DATE'])+'.nc')

In [ ]:
if mod_temp:
    for file in os.listdir(path):
        if file.startswith("backup_met_em.d01."):
            file_path=os.path.join(path, file)
            #Import metgrid file as xarray dataset
            ds=xr.open_dataset(file_path)
    
            ########################################
            # modification's to met_em* file below #
            #                                      #
            # Want to increase the temperature by  #
            # 5 deg C                              #
            ########################################
            ds_T = ds.TT
            ds_T = ds_T + 5
            ds['TT']=ds_T
    
            #Save new met_em files
            ds.to_netcdf(path+'met_em.d01.'+str(ds.attrs['SIMULATION_START_DATE'])+'.nc')
        else:
            pass

In [ ]:
if mod_rh:
    for file in os.listdir(path):
        if file.startswith("backup_met_em.d01."):
            file_path=os.path.join(path, file)
            #Import metgrid file as xarray dataset
            ds=xr.open_dataset(file_path)
    
            ########################################
            # modification's to met_em* file below #
            #                                      #
            # current configuration sets RH to be  #
            # 0 above the 10th metgrid level.      #
            # Can modify whatever variable at      #
            # whatever level you would like        #
            ########################################
            ds_rh=ds.RH
            #ds_rh[Time, num_metgrid_levels, south_north, west_east]
            ds_rh[:,10:, :, :]=ds_rh[ , , , ,].where(ds_rh[ , , , ,]<0,0)
            ds['RH']=ds_rh
    
            #Save new met_em files
            ds.to_netcdf(path+'met_em.d01.'+str(ds.attrs['SIMULATION_START_DATE'])+'.nc')
        else:
            pass

In [ ]:
### Import data to check that met_em* files were modified
### Initialization time is selected but any time could be chosen
ds_unmod = xr.open_dataset('backup_met_em.d01.2023-01-22_21:00:00.nc')
ds_mod = xr.open_dataset('met_em.d01.2023-01-22_21:00:00.nc')

### Remove dimension of time (length 1; only info for 1 time)
ds_unmod = ds_unmod.squeeze(dim='Time')
ds_mod = ds_mod.squeeze(dim='Time')

In [ ]:
##########################
# Plots for SNOWSTORM_01 #
# modify temperature (TT)#
##########################

In [ ]:
### Plot x-z cross section at a select y point
### Select south_north (y) point desired
if mod_temp:
    ## Plot figure
    cols = [0,1]
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12.3, 4.0), sharex=True)

    # Index data
    ds_unmod_plot = ds_unmod.isel(south_north=sn_num)
    ds_mod_plot = ds_mod.isel(south_north=sn_num)
    
    # axs[0] Unmodified RH
    axs[0].pcolormesh(ds_unmod_plot.TT - 273.15, cmap = temp_cmap , norm = temp_norm)
    
    # axs[1] Modified RH
    t_cb = axs[1].pcolormesh(ds_mod_plot.TT -273.15, cmap = temp_cmap, norm = temp_norm)
    
    # Colorbar
    cbar_axs = fig.add_axes([.92, 0.11, 0.03, 0.77])
    cb = fig.colorbar(t_cb, ax=axs[1],cax=cbar_axs)
    cb.set_label(r'Temperature ($^\circ$C)')
    
    # Plot labels & etc.
    axs[0].set_title(ds.attrs['SIMULATION_START_DATE']+' | south_north = '+str(sn_num)+'\nUnmodified met_em*', loc='left')
    axs[1].set_title('Modified met_em*', loc='left')
    
    axs[0].set_ylabel('number of metgrid levels (#)')
    
    for c in cols:
        axs[c].set_xlabel('west_east')
        axs[c].grid(True)
        
    plt.show()

In [ ]:
### Plot y-z cross section at a select x point
### Select west_east (x) point desired
if mod_temp:
    ## Plot figure
    cols = [0,1]
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12.3, 4.0), sharex=True)
    
    # Index data
    ds_unmod_plot = ds_unmod.isel(west_east=we_num)
    ds_mod_plot = ds_mod.isel(west_east=we_num)
    
    # axs[0] Unmodified RH
    axs[0].pcolormesh(ds_unmod_plot.TT - 273.15, cmap = temp_cmap, norm = temp_norm)
    
    # axs[1] Modified RH
    t_cb = axs[1].pcolormesh(ds_mod_plot.TT - 273.15, cmap = temp_cmap, norm = temp_norm)
    
    # Colorbar
    cbar_axs = fig.add_axes([.92, 0.11, 0.03, 0.77])
    cb = fig.colorbar(t_cb, ax=axs[1],cax=cbar_axs)
    cb.set_label(r'Temperature ($^\circ$C)')
    
    # Plot labels & etc.
    axs[0].set_title(ds.attrs['SIMULATION_START_DATE']+' | west_east = '+str(sn_num)+'\nUnmodified met_em*', loc='left')
    axs[1].set_title('Modified met_em*', loc='left')
    
    axs[0].set_ylabel('number of metgrid levels (#)')
    
    for c in cols:
        axs[c].set_xlabel('south_north')
        axs[c].grid(True)
    
    plt.show()

In [ ]:
### Plot variable at specified metgrid level
if mod_temp:
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12.3, 4.0), subplot_kw={'projection': proj})
    
    # Index data
    ds_unmod_plot = ds_unmod.isel(num_metgrid_levels=mgl_num)
    ds_mod_plot = ds_mod.isel(num_metgrid_levels=mgl_num)
    
    # Get lat and lon
    lon = ds_unmod.XLONG_M
    lat = ds_unmod.XLAT_M
    
    # axs[0] Unmodified temperature
    axs[0].pcolormesh(lon, lat, ds_unmod_plot.TT - 273.15, cmap=temp_cmap, norm = temp_norm_map, shading='auto', transform=proj)
    
    # axs[1] Modified temperature
    t_cb = axs[1].pcolormesh(lon, lat, ds_mod_plot.TT - 273.15, cmap=temp_cmap, norm = temp_norm_map, shading='auto', transform=proj)
    
    # Colorbar
    cbar_axs = fig.add_axes([.92, 0.11, 0.03, 0.77])
    cb = fig.colorbar(t_cb, ax=axs[1],cax=cbar_axs)
    cb.set_label(r'Temperature ($^\circ$C)')
    
    # Plot labels & etc.
    axs[0].set_title(ds.attrs['SIMULATION_START_DATE'] +' | num_metgrid_levels = ' + str(mgl_num) +'\nUnmodified met_em*', loc='left')
    axs[1].set_title('Modified met_em*', loc='left')
    
    # Plot labels & etc.
    for ax in axs:
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.add_feature(cfeature.STATES, linewidth=0.7, edgecolor='black')
        ax.add_feature(cfeature.BORDERS, linewidth=0.7)
        ax.coastlines(resolution='10m', linewidth=0.7)
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
        
        
        gl = ax.gridlines(draw_labels=True, linewidth=0.7, color='black', alpha=0.5, linestyle='-')
        gl.top_labels = False
        gl.right_labels = False
    
    plt.show()

In [ ]:
##########################
# Plots for SNOWSTORM_02 #
# modify RH              #
##########################

In [ ]:
### Plot x-z cross section at a select y point
### Select south_north (y) point desired
if mod_rh:
    ## Plot figure
    cols = [0,1]
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12.3, 4.0), sharex=True)
    # Index data
    ds_unmod_plot = ds_unmod.isel(south_north=sn_num)
    ds_mod_plot = ds_mod.isel(south_north=sn_num)
    
    # axs[0] Unmodified RH
    axs[0].pcolormesh(ds_unmod_plot.RH, cmap = rh_cmap, norm = rh_norm)
    
    # axs[1] Modified RH
    rh_cb = axs[1].pcolormesh(ds_mod_plot.RH, cmap = rh_cmap, norm = rh_norm)
    
    # Colorbar
    
    cbar_axs = fig.add_axes([.92, 0.11, 0.03, 0.77])
    cb = fig.colorbar(rh_cb, ax=axs[1],cax=cbar_axs)
    cb.set_label('RH (%)')
    
    # Plot labels & etc.
    axs[0].set_title(ds.attrs['SIMULATION_START_DATE']+' | south_north = '+str(sn_num)+'\nUnmodified met_em*', loc='left')
    axs[1].set_title('Modified met_em*', loc='left')
    
    axs[0].set_ylabel('number of metgrid levels (#)')
    
    for c in cols:
        axs[c].set_xlabel('west_east')
        axs[c].grid(True)

    plt.show()

In [ ]:
### Plot y-z cross section at a select x point
### Select west_east (x) point desired
if mod_rh:
    ## Plot figure
    cols = [0,1]
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12.3, 4.0), sharex=True)
    
    # Index data
    ds_unmod_plot = ds_unmod.isel(west_east=we_num)
    ds_mod_plot = ds_mod.isel(west_east=we_num)
    
    # axs[0] Unmodified RH
    axs[0].pcolormesh(ds_unmod_plot.RH, cmap = rh_cmap, norm = rh_norm)
    
    # axs[1] Modified RH
    rh_cb = axs[1].pcolormesh(ds_mod_plot.RH, cmap = rh_cmap, norm = rh_norm)
    
    # Colorbar
    cbar_axs = fig.add_axes([.92, 0.11, 0.03, 0.77])
    cb = fig.colorbar(rh_cb, ax=axs[1],cax=cbar_axs)
    cb.set_label('RH (%)')
    
    # Plot labels & etc.
    axs[0].set_title(ds.attrs['SIMULATION_START_DATE']+' | west_east = '+str(sn_num)+'\nUnmodified met_em*', loc='left')
    axs[1].set_title('Modified met_em*', loc='left')
    
    axs[0].set_ylabel('number of metgrid levels (#)')
    
    for c in cols:
        axs[c].set_xlabel('south_north')
        axs[c].grid(True)
        
    plt.show()

In [ ]:
### Plot variable at specified metgrid level
if mod_rh:
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12.3, 4.0), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Set range for CB
    rh_norm = colors.Normalize(vmin=0, vmax=100)
    
    # Index data
    ds_unmod_plot = ds_unmod.isel(num_metgrid_levels=mgl_num)
    ds_mod_plot = ds_mod.isel(num_metgrid_levels=mgl_num)
    
    # Get lat and lon
    lon = ds_unmod.XLONG_M
    lat = ds_unmod.XLAT_M
    
    # axs[0] Unmodified RH
    axs[0].pcolormesh(lon, lat, ds_unmod_plot.RH, cmap=rh_cmap, norm = rh_norm_map, shading='auto', transform=proj)
    
    # axs[1] Modified RH
    rh_cb = axs[1].pcolormesh(lon, lat, ds_mod_plot.RH, cmap=rh_cmap, norm = rh_norm_map, shading='auto', transform=proj)
    
    # Colorbar
    cbar_axs = fig.add_axes([.92, 0.11, 0.03, 0.77])
    cb = fig.colorbar(rh_cb, ax=axs[1],cax=cbar_axs)
    cb.set_label('RH (%)')
    
    # Plot labels & etc.
    axs[0].set_title(ds.attrs['SIMULATION_START_DATE'] + ' | num_metgrid_levels = ' + str(mgl_num) + '\nUnmodified met_em*', loc='left')
    
    axs[1].set_title('Modified met_em*', loc='left')
    
    # Plot labels & etc.
    for ax in axs:
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.add_feature(cfeature.STATES, linewidth=0.7, edgecolor='black')
        ax.add_feature(cfeature.BORDERS, linewidth=0.7)
        ax.coastlines(resolution='10m', linewidth=0.7)
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
        
        gl = ax.gridlines(draw_labels=True, linewidth=0.7, color='black', alpha=0.5, linestyle='-')
        gl.top_labels = False
        gl.right_labels = False
    
    plt.show()